In [1]:
import copy
import pickle

from pyflink.common import Row, Types
from pyflink.datastream import (
    KeyedProcessFunction,
    RuntimeContext,
    StreamExecutionEnvironment,
)
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.table import EnvironmentSettings, StreamTableEnvironment

In [2]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
t_env = StreamTableEnvironment.create(env, environment_settings=settings)

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
t_env.execute_sql("DROP TABLE IF EXISTS klines_source")
t_env.execute_sql("""
CREATE TABLE klines_source (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE
) WITH (
    'connector' = 'filesystem',
    'path' = '/workspace/output/klines/ADAUSDT/2025-09-27',
    'format' = 'csv'
)
""")

In [ ]:
def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10**decimals
    return float(int(x * factor + 0.5)) / factor

In [4]:
class EngulfingPatternFunction(KeyedProcessFunction):
    def open(self, runtime_context: RuntimeContext):
        self.prev_row_state = runtime_context.get_state(
            ValueStateDescriptor("prev_row_state", Types.PICKLED_BYTE_ARRAY())
        )

    def calc_ema(self, close_price, period, ema_state, buffer_state):
        if close_price is None:
            return None
        k = 2 / (period + 1)
        if ema_state is None:
            buffer_state.append(close_price)
            if len(buffer_state) < period:
                return None
            ema = sum(buffer_state) / len(buffer_state)
            buffer_state.clear()
            return ema
        else:
            return (close_price - ema_state) * k + ema_state

    def detect_trend(self, ema7, ema20):
        if ema7 is None or ema20 is None:
            return None
        if ema7 > ema20:
            return "uptrend"
        elif ema7 < ema20:
            return "downtrend"
        else:
            return None

    def detect_engulfing(self, current, previous, trend):
        if not previous or not trend:
            return None
        open_price_prev = previous["open_price"]
        close_price_prev = previous["close_price"]
        open_price = current["open_price"]
        close_price = current["close_price"]

        if (
            close_price_prev < open_price_prev
            and close_price > open_price
            and open_price < close_price_prev
            and close_price > open_price_prev
            and trend == "downtrend"
        ):
            return "bullish engulfing"

        if (
            close_price_prev > open_price_prev
            and close_price < open_price
            and open_price > close_price_prev
            and close_price < open_price_prev
            and trend == "uptrend"
        ):
            return "bearish engulfing"

        return None

    def process_element(self, value, ctx):
        prev_row_bytes = self.prev_row_state.value()
        prev_row_state = pickle.loads(prev_row_bytes) if prev_row_bytes else {}
        buffer7_state = copy.deepcopy(prev_row_state.get("buffer7_state", []))
        buffer20_state = copy.deepcopy(prev_row_state.get("buffer20_state", []))

        ema7 = self.calc_ema(
            value["close_price"], 7, prev_row_state.get("ema7"), buffer7_state
        )
        ema20 = self.calc_ema(
            value["close_price"], 20, prev_row_state.get("ema20"), buffer20_state
        )

        trend = self.detect_trend(ema7, ema20)
        pattern = self.detect_engulfing(value.as_dict(), prev_row_state, trend)

        new_state = {
            **value.as_dict(),
            "ema7": ema7,
            "ema20": ema20,
            "trend": trend,
            "pattern": pattern,
            "buffer7_state": buffer7_state,
            "buffer20_state": buffer20_state,
        }
        self.prev_row_state.update(pickle.dumps(new_state))

        yield Row(
            **value.as_dict(),
            ema7=round_half_up(ema7, 4) if ema7 else None,
            ema20=round_half_up(ema20, 4) if ema20 else None,
            trend=trend,
            engulfing_pattern=pattern,
        )

In [5]:
klines_stream = t_env.to_data_stream(t_env.from_path("klines_source")).map(
    lambda r: Row(**r.as_dict(), symbol="ADAUSDT")
)

typeinfo = Types.ROW_NAMED(
    [
        "window_start",
        "window_end",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
        "ema7",
        "ema20",
        "trend",
        "engulfing_pattern",
        "symbol",
    ],
    [
        Types.SQL_TIMESTAMP(),
        Types.SQL_TIMESTAMP(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.STRING(),
        Types.STRING(),
        Types.STRING(),
    ],
)

engulfing_stream = klines_stream.key_by(lambda x: x["symbol"]).process(
    EngulfingPatternFunction(), output_type=typeinfo
)

In [3]:
t_env.execute_sql("DROP TABLE IF EXISTS engulfing_sink")
t_env.execute_sql("""
CREATE TABLE engulfing_sink (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE,
    ema7 DOUBLE,
    ema20 DOUBLE,
    trend STRING,
    engulfing_pattern STRING,
    symbol STRING
) WITH (
    'connector' = 'filesystem',
    'path' = '/workspace/output/engulfing',
    'format' = 'csv',
    'csv.null-literal' = ''
)
""")

In [6]:
t_env.drop_temporary_view("engulfing_stream")
t_env.create_temporary_view(
    "engulfing_stream", t_env.from_data_stream(engulfing_stream)
)
t_env.execute_sql("INSERT INTO engulfing_sink SELECT * FROM engulfing_stream")

In [4]:
import pandas as pd
pd.set_option("display.max_rows", 100)       # Show up to 100 rows
pd.set_option("display.max_columns", None)   # Show all columns
pd.set_option("display.width", 0)            # Auto-detect console width
pd.set_option("display.max_colwidth", None)  # Don’t truncate cell contents
t_env.from_path("engulfing_sink").to_pandas().head(100)

,window_start,window_end,open_price,high_price,low_price,close_price,volume,ema7,ema20,trend,engulfing_pattern,symbol
0,2025-09-27 00:00:00,2025-09-27 00:15:00,0.7918,0.7918,0.7897,0.7901,465779.1,NaN,NaN,None,None,ADAUSDT
1,2025-09-27 00:15:00,2025-09-27 00:30:00,0.7902,0.7924,0.7900,0.7911,163062.6,NaN,NaN,None,None,ADAUSDT
2,2025-09-27 00:30:00,2025-09-27 00:45:00,0.7912,0.7920,0.7905,0.7913,202817.4,NaN,NaN,None,None,ADAUSDT
3,2025-09-27 00:45:00,2025-09-27 01:00:00,0.7913,0.7927,0.7899,0.7923,434212.2,NaN,NaN,None,None,ADAUSDT
4,2025-09-27 01:00:00,2025-09-27 01:15:00,0.7924,0.7927,0.7902,0.7913,555784.6,NaN,NaN,None,None,ADAUSDT
5,2025-09-27 01:15:00,2025-09-27 01:30:00,0.7912,0.7922,0.7882,0.7897,550528.6,NaN,NaN,None,None,ADAUSDT
6,2025-09-27 01:30:00,2025-09-27 01:45:00,0.7898,0.7905,0.7886,0.7899,405461.3,0.7908,NaN,None,None,ADAUSDT
7,2025-09-27 01:45:00,2025-09-27 02:00:00,0.7900,0.7907,0.7898,0.7903,127489.5,0.7907,NaN,None,None,ADAUSDT
8,2025-09-27 02:00:00,2025-09-27 02:15:00,0.7903,0.7910,0.7890,0.7909,387130.3,0.7907,NaN,None,None,ADAUSDT
9,2025-09-27 02:15:00,2025-09-27 02:30:00,0.7909,0.7912,0.7885,0.7896,273607.2,0.7905,NaN,None,None,ADAUSDT


In [10]:
!jupyter nbconvert --to script test_transform_job_pattern_two.ipynb

I0000 00:00:1760455199.818734     480 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


[NbConvertApp] Converting notebook test_transform_job_pattern_two.ipynb to script
[NbConvertApp] Writing 8408 bytes to test_transform_job_pattern_two.py


In [5]:
# klines_source = t_env.from_path("klines_source")
# klines_source.execute().print()
# klines_source.to_pandas().head(20)

# printed = {"done": False}
# def inspect_types(r):
#     if not printed["done"]:
#         print({name: type(value).__name__ for name, value in zip(r._fields, r)})
#         printed["done"] = True
#     return r
# klines_stream = t_env.to_data_stream(t_env.from_path("klines_source"))
# klines_stream.map(inspect_types)
# klines_stream.get_type()
# env.execute("Print Klines Source")

# self.ema7_state = runtime_context.get_state(
#     ValueStateDescriptor("ema7", Types.DOUBLE())
# )
# self.ema20_state = runtime_context.get_state(
#     ValueStateDescriptor("ema20", Types.DOUBLE())
# )
# self.buffer7_state = runtime_context.get_list_state(
#     ListStateDescriptor("buffer7", Types.DOUBLE())
# )
# self.buffer20_state = runtime_context.get_list_state(
#     ListStateDescriptor("buffer20", Types.DOUBLE())
# )

# prev_row_desc = ValueStateDescriptor(
#     "prev_row",
#     Types.ROW_NAMED(
#         ["open_price", "close_price", "volume"],
#         [Types.DOUBLE(), Types.DOUBLE(), Types.DOUBLE()]
#     )
# )
# self.prev_row_state = runtime_context.get_state(prev_row_desc)

# import json
# self.prev_row_state = runtime_context.get_state(
#     ValueStateDescriptor("prev_row_json", Types.STRING())
# )
# # Save
# self.prev_row_state.update(json.dumps(value.as_dict()))
# # Load
# prev_row = json.loads(self.prev_row_state.value()) if self.prev_row_state.value() else None